# Multi-Agent 대표 패턴 학습 실습

---

## 학습 개요

### 학습 주제
- **Multi-Agent 시스템의 필요성**: 단일 Agent의 한계를 이해하고 Multi-Agent가 필요한 상황을 판단
- **Multi-Agent 대표 패턴**: Planner-Worker, Reflection 등 협업 패턴의 구조와 특징 학습
- **LangGraph 워크플로우 구성**: StateGraph를 활용한 에이전트 간 상태 공유 및 흐름 제어

### 학습 목표
- Multi-Agent 시스템의 구조를 이해하고, 상황에 적합한 패턴을 선택하여 구현할 수 있다
  - 단일 Agent의 한계를 설명하고 Multi-Agent가 필요한 상황을 판단할 수 있다
  - Planner-Worker 역할 분리 구조를 이해하고 구현할 수 있다
  - Reflection 패턴과 성과 평가를 통해 결과 품질을 향상시키고 정량적으로 확인할 수 있다
  - LangGraph StateGraph를 사용하여 Agent 간 상태 공유 워크플로우를 구성할 수 있다
  - Multi-Agent 대표 패턴 5가지의 특징과 적합한 상황을 비교 설명할 수 있다

### 핵심 개념
- **Multi-Agent 시스템**: 여러 Agent가 역할을 분담하여 협업하는 구조. 각 Agent는 독립적인 LLM 호출과 프롬프트를 가지며, State를 통해 정보를 공유한다. 단일 Agent 대비 복잡한 작업에서 품질과 신뢰성이 향상된다.
- **Planner Agent**: 사용자 요청을 분석하고 작업 계획을 수립하는 역할. "무엇을 해야 하는지"를 정의하며, 직접 실행하지 않고 계획만 수립한다. 계획의 명확성이 전체 결과 품질을 좌우한다.
- **Worker Agent**: Planner의 계획에 따라 실제 작업을 수행하는 역할. 계획의 각 단계를 구체적인 결과물로 변환한다. 전문화된 프롬프트로 각 단계에 집중한다.
- **Reflection 패턴**: 결과를 자기 검토하여 품질을 향상시키는 루프. Agent가 생성한 결과를 다시 평가하고 누락/오류를 보완한다. 원본 요청 충족 여부를 체계적으로 검토한다.
- **StateGraph**: LangGraph에서 제공하는 상태 기반 워크플로우 정의 클래스. 노드(Agent)와 엣지(흐름)로 구성되며, State 객체를 통해 Agent 간 정보를 전달한다. TypedDict를 상속하여 State 스키마를 정의한다.
- **성과 평가 (Reward Model)**: Multi-Agent 결과 품질을 정량적으로 측정하는 평가 함수. Planner 계획의 완성도, Worker 실행의 충실도, 전체 요청 충족도를 평가한다.

### 선행 지식
- Python 함수 정의, 클래스 기본 문법, 딕셔너리 조작 (`dict.get()`, `**dict` 언패킹)
- 4-2(1) 실습 완료 (ReAct Agent 구현 경험, Tool 사용 이해)
- LLM API 호출 방식 (`llm.invoke()`), 프롬프트 엔지니어링 기초
- LangChain 기본 사용법 (ChatOpenAI, 메시지 포맷)

---

## 실습 구성

### 학습 방향

- **실습 구성 방식**
  - 문제와 정답 코드가 병렬로 제공되며, 각 단계별 TODO 영역을 채우며 학습자가 직접 구현

- **Required Package**
  - `langchain>=0.3.0`: LLM 체인 프레임워크
  - `langchain-openai>=0.2.0`: OpenAI 모델 연동
  - `langgraph>=0.2.0`: Multi-Agent 워크플로우
  - `python-dotenv>=1.0.0`: 환경 변수 관리

- **실행 환경**
  - Python 3.10 이상 권장 (LangGraph의 타입 힌팅 기능 활용)

- **Step 요약**
  - **Step 1 (5분)**: 환경 설정 — 패키지 설치, API 키 설정
  - **Step 2 (10분)**: 단일 Agent의 한계 체감 — 복잡한 작업을 단일 Agent로 시도
  - **Step 3 (5분)**: Multi-Agent 패턴 개요 — 5가지 대표 패턴 학습
  - **Step 4 (15분)**: Planner-Worker 구현 — 계획 수립과 실행 분리
  - **Step 5 (15분)**: Reflection 패턴 추가 — 자기 검토 루프로 품질 향상
  - **Step 6 (10분)**: 성과 평가 — 결과 품질을 정량적으로 측정

### 문제 설명

- **문제 개요**: 이 실습은 Multi-Agent 시스템의 필요성을 이해하고, Planner-Worker 및 Reflection 패턴을 구현하기 위해 설계되었습니다. 학습자는 LangGraph StateGraph를 사용하여 에이전트 간 상태 공유 워크플로우를 구성하고, 최종적으로 Multi-Agent 대표 패턴 5가지의 특징과 적합한 상황을 비교 설명할 수 있어야 합니다.
- **요구사항 요약**
  - TODO 1: 단일 Agent로 복잡한 작업을 시도하여 한계 체감
  - TODO 2: Planner Node를 구현하여 단계별 계획 수립
  - TODO 3: Worker Node를 구현하여 계획을 실행
  - TODO 4: Reflection Node를 구현하여 결과 검토 및 개선
  - TODO 5: 성과 평가 함수를 구현하여 결과 품질 정량 측정
- **주의사항**
  - LLM API 키가 올바르게 설정되어야 합니다
  - 각 Agent의 프롬프트는 역할을 명확히 분리해야 합니다

---

# Step 1: 환경 설정

### Setup: 라이브러리 설치 및 API Key 설정

In [1]:
%%capture
# 필요한 패키지 설치
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langgraph>=0.2.0" "python-dotenv>=1.0.0" -q

In [2]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
if not os.environ.get("GMS_KEY"):
    os.environ["GMS_KEY"] = input("🔑 GMS_KEY를 입력하세요: ")
    
print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.environ["GMS_KEY"],
    base_url="https://gms.ssafy.io/gmsapi/api.openai.com/v1/"
)

In [4]:
# 모델 연결 테스트
response = llm.invoke("3일 서울 여행 계획을 간단히 요약해줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})")

LLM 응답:
3일 서울 여행 — 간단 요약

Day 1 (전통·경복궁 코스)
- 오전: 경복궁·국립민속박물관 관람 → 광화문 광장 사진
- 점심: 북촌·삼청동에서 한식(비빔밥, 한정식)
- 오후: 북촌 한옥마을 산책 → 인사동(기념품, 찻집)
- 저녁: 청계천 산책 후 명동 쇼핑·거리음식
- 팁: 경복궁 한복 착용 시 입장 무료

Day 2 (트렌디·젊은이 코스)
- ...

✅ 모델 연결 확인 완료 (gpt-5-mini)


---

# Step 2: 단일 Agent의 한계 체감

### Concept Check: 단일 Agent의 특징과 한계

**왜 이 단계가 필요한가요?**  
Multi-Agent의 필요성을 이해하려면 먼저 단일 Agent의 한계를 직접 체감해야 합니다. 이론적인 설명보다 실제로 복잡한 작업을 단일 Agent에게 맡겨보고 결과의 문제점을 확인하는 것이 효과적입니다.

**단일 Agent**는 하나의 LLM이 모든 작업을 처리합니다:
- 장점: 구현이 간단함
- 단점: 복잡한 작업에서 혼란 발생, 작업 누락, 품질 저하

복잡한 작업 예시:
> "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산도 포함해서."

이 작업은 여러 하위 작업을 포함합니다:
1. 여행 일정 계획
2. 맛집 추천
3. 관광지 추천
4. 예산 계산

#### 단일 Agent의 한계 시각화

```
┌─────────────────────────────────────────────────────────┐
│                    단일 Agent                            │
│  ┌─────────────────────────────────────────────────┐   │
│  │ 복잡한 요청                                       │   │
│  │ "3일 서울 여행 + 맛집 + 관광지 + 예산"            │   │
│  └─────────────────────────────────────────────────┘   │
│                         ↓                               │
│  ┌─────────────────────────────────────────────────┐   │
│  │ 하나의 LLM이 모든 작업을 한 번에 처리             │   │
│  │ - 일정 계획 + 맛집 + 관광지 + 예산 동시 처리      │   │
│  └─────────────────────────────────────────────────┘   │
│                         ↓                               │
│  ┌─────────────────────────────────────────────────┐   │
│  │ ⚠️ 발생 가능한 문제                               │   │
│  │ • 일부 요청 누락 (예: 예산 계산 생략)             │   │
│  │ • 정보 깊이 부족 (예: 맛집 가격대 누락)           │   │
│  │ • 일관성 문제 (예: 동선이 비효율적)               │   │
│  └─────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

단일 Agent가 이 모든 작업을 한 번에 처리하면 어떻게 될까요?

### TODO 1: 단일 Agent로 복잡한 작업 시도

- **요구사항**: 단일 Agent에게 복잡한 여행 계획 작업을 요청하고 결과를 확인합니다.
- **입력**: 
  - `complex_request` (str): 복잡한 여행 계획 요청 문자열
- **출력**: 
  - `response` (AIMessage): LLM의 응답 객체, `response.content`로 텍스트 접근
- **예상 결과**: 여행 계획이 출력되지만, 일부 정보 누락 또는 체계성 부족 가능
- **확인 방법**: `print(response.content)`로 출력 후 요청 사항 충족 여부 체크
- **힌트**: `llm.invoke()` 메서드를 사용합니다.

In [5]:
# TODO: 단일 Agent로 복잡한 작업 시도

# 복잡한 작업 요청
complex_request = """
3일 서울 여행 계획을 세워줘.
- 각 날짜별 상세 일정
- 점심, 저녁 맛집 추천 (위치, 메뉴, 가격대)
- 관광지 추천 (입장료, 소요시간)
- 전체 예산 계산
"""

# TODO: llm.invoke()를 사용하여 단일 Agent로 처리하세요
# response = ???

response = llm.invoke(complex_request)

if response is not None:
    print("=== 단일 Agent 응답 ===")
    print(response.content)
else:
    print("=== 코드를 완성해주세요 ===")

=== 단일 Agent 응답 ===
좋습니다. 3일간의 서울 여행(혼자 기준, 중간 수준 예산) 일정을 아래와 같이 제안합니다. 원하시면 예산·관심사(음식 취향, 쇼핑 여부, 숙소 등)에 맞춰 맞춤 조정해 드릴게요.

요약(핵심)
- 숙소: 시내 중심(종로·명동·홍대·강남 등) 3박(1인) 가정: 약 100,000원/박 가정
- 교통: T-money 카드 사용(지하철·버스) + 택시 약간
- 식비: 점심·저녁 중심(아침은 호텔/카페 또는 간단히) 
- 관광지는 입장료/소요시간 표기
- 전체 예산(대략): 경제형 350,000원, 중간형 540,000원, 고급형 1,100,000원(아래 세부 계산 참조)

Day 1 — 전통과 도심(경복궁 → 북촌 → 인사동 → 명동/남산)
- 09:00 경복궁 입장(권장 시간: 09:00~11:00)
  - 활동: 경회루/근정전 관람, 경복궁 중앙·국립민속박물관(일부 무료)
  - 입장료: 약 3,000원 (성인 기준, 한복 착용 시 무료 혜택이 있는 경우 확인)
  - 소요시간: 1.5~2시간
- 11:30 토속촌 삼계탕(점심)
  - 위치: 경복궁(광화문) 근처
  - 메뉴: 삼계탕(인삼 닭백숙) — 14,000~19,000원
  - 분위기: 전통 한식, 체력 보충용
- 13:00 북촌 한옥마을 산책
  - 소요시간: 1~1.5시간
  - 입장료: 무료(개별 한옥 박물관은 유료)
- 14:30 인사동 거리 & 전통 찻집
  - 활동: 기념품, 전통 공예품 구경, 찻집(전통 다과)
  - 소요시간: 1~2시간
- 17:00 명동 이동 & 쇼핑
  - 활동: 화장품·패션 쇼핑, 길거리 먹거리
- 19:00 명동교자(저녁)
  - 위치: 명동
  - 메뉴: 칼국수, 만두 등 — 8,000~12,000원
- (옵션) 21:00 남산·N서울타워 전망대
  - 입장료: 전망대 약 11,000원(성인) — 케이블카는 별도 요금(왕복 약 9,000원)
  - 소요시간: 1~1.5시간
- 이동시간: 지하철/버스 각 구간 15~30분

Da

### Test: 단일 Agent의 한계 확인

위 결과를 확인해보세요:
- [ ] 모든 요청 사항이 포함되었나요? (일정, 맛집, 관광지, 예산)
- [ ] 정보가 체계적으로 정리되었나요? (날짜별, 카테고리별)
- [ ] 예산 계산이 정확한가요? (숙박+식비+교통+입장료 합계)
- [ ] 맛집과 관광지 정보가 구체적인가요? (위치, 가격, 소요시간)

**확인 기준**:
- 4가지 요청 사항 중 1개 이상 누락 → 단일 Agent 한계 체감
- 정보 간 연결성 부족 (예: 동선 고려 없는 일정) → 체계성 문제 확인

**예상 출력 예시** (한계가 드러나는 경우):
```
✓ 일정: "Day 1: 경복궁, 명동..." (있음)
✓ 관광지: "경복궁, 북촌한옥마을..." (있음)
△ 맛집: "광장시장 빈대떡" (가격대 누락)
✗ 예산: 전체 합계 없음 또는 부정확
✗ 동선: 강남→종로→잠실 (비효율적 이동)
```

**체감 포인트**: "하나의 Agent가 모든 걸 하려니 복잡하고, 일부 정보가 누락되거나 품질이 일정하지 않네"

---

# Step 3: Multi-Agent 패턴 개요

### Concept Check: Multi-Agent 시스템이란?

**왜 이 단계가 필요한가요?**  
Step 2에서 단일 Agent의 한계를 체감했으니, 이제 그 해결책인 Multi-Agent 시스템의 다양한 패턴을 학습합니다. 어떤 상황에 어떤 패턴이 적합한지 이해해야 실제 프로젝트에서 올바른 선택을 할 수 있습니다.

**Multi-Agent 시스템**은 여러 Agent가 역할을 나누어 협업하는 구조입니다.

#### Single Agent vs Multi-Agent

| 구분 | Single Agent | Multi-Agent |
|------|--------------|-------------|
| 구조 | 하나의 LLM이 모든 작업 처리 | 여러 LLM이 역할 분담 |
| 장점 | 구현 간단 | 복잡한 작업 처리 가능, 품질 향상 |
| 단점 | 복잡한 작업에서 한계 | 구현 복잡도 증가 |
| 적합한 상황 | 단순한 Q&A | 다단계 작업, 전문 영역 분리 필요 시 |

#### Multi-Agent 대표 패턴 5가지

| 패턴 | 설명 | 적합한 상황 | 난이도 |
|------|------|-------------|--------|
| **Planner-Worker** | 계획 수립 → 실행 분리 | 순차적 다단계 작업 | 하 |
| **Supervisor** | 관리자가 여러 Worker 조율 | 병렬 처리 필요한 독립 작업 | 중 |
| **Debate/Discussion** | 여러 Agent가 토론 후 결론 | 다양한 관점이 필요한 의사결정 | 중 |
| **Reflection** | 자기 검토/개선 루프 | 품질 보장이 중요한 작업 | 하 |
| **Hierarchical** | 계층적 위임 구조 | 대규모 복잡한 프로젝트 | 상 |

이번 실습에서는 **Planner-Worker**와 **Reflection** 패턴을 구현합니다.

#### LangGraph란?

**LangGraph**는 LangChain 팀이 개발한 워크플로우 오케스트레이션 라이브러리입니다:
- **StateGraph**: 상태 기반 그래프로 에이전트 흐름 정의. `TypedDict`를 상속하여 State 스키마 정의
- **Node**: 각 에이전트가 수행하는 작업 단위. `def node_func(state) -> state` 형태의 함수
- **Edge**: 노드 간 연결 (흐름 정의). `add_edge("node_a", "node_b")`로 순차 연결
- **State**: 에이전트 간 공유되는 상태 정보. 딕셔너리 형태로 전달되며 각 노드가 업데이트

#### LangGraph State 흐름도

```
┌──────────────────────────────────────────────────────────────────┐
│                     LangGraph State 흐름                          │
├──────────────────────────────────────────────────────────────────┤
│                                                                   │
│  [State 객체]                                                     │
│  ┌─────────────────────────────────────────────┐                 │
│  │ user_request: "3일 서울 여행..."             │                 │
│  │ plan: ""                                     │                 │
│  │ result: ""                                   │                 │
│  └─────────────────────────────────────────────┘                 │
│           │                                                       │
│           ▼                                                       │
│  ┌─────────────────┐                                             │
│  │  Planner Node   │ ─── State 읽기: user_request                │
│  │  (계획 수립)     │ ─── State 쓰기: plan = "Step 1..."         │
│  └─────────────────┘                                             │
│           │                                                       │
│           ▼ (Edge: planner → worker)                             │
│  ┌─────────────────┐                                             │
│  │  Worker Node    │ ─── State 읽기: user_request, plan          │
│  │  (실행)         │ ─── State 쓰기: result = "Day 1: ..."       │
│  └─────────────────┘                                             │
│           │                                                       │
│           ▼                                                       │
│  [최종 State]                                                     │
│  ┌─────────────────────────────────────────────┐                 │
│  │ user_request: "3일 서울 여행..."             │                 │
│  │ plan: "Step 1: 일정...\nStep 2: 맛집..."    │ ← Planner 결과  │
│  │ result: "Day 1: 경복궁(09:00)..."           │ ← Worker 결과   │
│  └─────────────────────────────────────────────┘                 │
│                                                                   │
└──────────────────────────────────────────────────────────────────┘
```

### Guided Build: LangGraph State 및 기본 구조

LangGraph를 사용하려면 먼저 **State(상태)**를 정의해야 합니다. 
State는 에이전트들이 공유하는 정보를 담는 컨테이너입니다.

In [6]:
# LangGraph import 및 State 정의
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    """Multi-Agent 시스템의 상태 정의"""
    user_request: str     # 사용자 요청
    plan: str             # Planner가 생성한 계획
    result: str           # Worker가 생성한 결과
    reflection: str       # Reflection Agent의 검토 결과
    final_result: str     # 최종 결과
    reflection_count: int # Reflection 반복 횟수 (무한 루프 방지)
    messages: List[dict]  # 에이전트 간 메시지 기록

# State 구조 확인
print("AgentState 필드:")
print("  - user_request: 사용자 요청 문자열")
print("  - plan: Planner Agent의 계획")
print("  - result: Worker Agent의 실행 결과")
print("  - reflection: Reflection Agent의 검토 결과")
print("  - final_result: 최종 결과")
print("  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)")
print("  - messages: 에이전트 간 메시지 기록")

AgentState 필드:
  - user_request: 사용자 요청 문자열
  - plan: Planner Agent의 계획
  - result: Worker Agent의 실행 결과
  - reflection: Reflection Agent의 검토 결과
  - final_result: 최종 결과
  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)
  - messages: 에이전트 간 메시지 기록


---

# Step 4: LangGraph로 Planner-Worker 구현

### Concept Check: 역할별 프롬프트 설계

**왜 이 단계가 필요한가요?**  
이제 Multi-Agent 패턴 중 가장 기본적인 Planner-Worker를 직접 구현합니다. "계획"과 "실행"을 분리함으로써 각 Agent가 자신의 역할에 집중할 수 있고, 결과의 체계성이 향상됩니다.

각 Agent는 고유한 역할과 프롬프트를 가집니다:

- **Planner**: 요청을 분석하고 단계별 계획을 수립. "무엇을", "어떤 순서로" 할지 정의
- **Worker**: 계획을 받아 각 단계를 실행. 구체적인 정보와 결과물 생성

LangGraph에서는 각 Agent를 **노드(Node)**로 정의하고, State를 통해 정보를 전달합니다.

#### Agent 간 메시지 흐름 예시
```
[User] "3일 서울 여행 계획"
    ↓
[Planner] State 읽기: user_request
    ↓ LLM 호출
[Planner] State 쓰기: plan = "Step 1: 일정 계획\nStep 2: 맛집 추천..."
    ↓
[Worker] State 읽기: user_request, plan
    ↓ LLM 호출
[Worker] State 쓰기: result = "Day 1: 경복궁(09:00-12:00)..."
```

### TODO 2: Planner Node 구현

- **요구사항**: 사용자 요청을 분석하고 단계별 계획을 수립하는 Planner Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["user_request"]` (str): "3일 서울 여행 계획..." 형태의 요청
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `plan` (str): "Step 1: 일정 계획\nStep 2: 맛집 추천..." 형태의 계획
    - `messages` (List[dict]): 메시지 로그에 Planner 결과 추가
- **예상 결과**: 3~5개의 Step으로 구성된 계획 문자열
- **확인 방법**: `print(result_state['plan'])`으로 계획 출력 확인
- **힌트**: State를 받아서 업데이트된 State를 반환하는 함수로 구현합니다. `{**state, "plan": plan}`으로 State 업데이트

In [7]:
# TODO: Planner Node 구현

def planner_node(state: AgentState) -> AgentState:
    """
    Planner Node: 사용자 요청을 분석하고 단계별 계획을 수립합니다.
    
    Args:
        state: 현재 AgentState
    
    Returns:
        업데이트된 AgentState (plan 필드 추가)
    """
    user_request = state["user_request"]
    
    # TODO: Planner 프롬프트를 작성하세요
    # - 사용자 요청을 분석하고 단계별 계획을 수립하도록 지시
    # - "계획만 수립하고, 직접 실행하지 말 것"을 명시
    # - 출력 형식: Step 1: [작업], Step 2: [작업], ...
    planner_prompt = f"""
    # TODO: 프롬프트 작성
    당신은 작업 계획을 수립하는 Planner Agent입니다.

    사용자 요청을 분석하고, 수행해야 할 작업을 단계별로 나열하세요.
    각 단계는 구체적이고 실행 가능해야 합니다.

    중요: 계획만 수립하세요. 직접 실행하지 마세요.

    출력 형식:
    Step 1: [작업 설명]
    Step 2: [작업 설명]
    ...

    ---
    사용자 요청: {user_request}
    """
    
    # LLM 호출
    response = llm.invoke(planner_prompt)
    plan = response.content
    
    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Planner",
        "receiver": "Worker",
        "content": plan,
        "task_type": "plan"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "plan": plan,  # TODO: plan 변수로 교체
        "messages": messages
    }

# 테스트
test_state = {
    "user_request": "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산 포함.",
    "plan": "",
    "result": "",
    "messages": []
}

# TODO 완료 후 테스트
result_state = planner_node(test_state)
if result_state['plan'] is not None:
    print("=== Planner Node 결과 ===")
    print(f"계획:\n{result_state['plan']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== Planner Node 결과 ===
계획:
Step 1: 사용자 정보 확인 요청
- 여행 날짜(출발/도착), 인원 수, 연령대(어린이/노인 등), 출발지(공항/역), 숙박 선호지역, 여행 스타일(느긋/빡빡/쇼핑 위주/미식 위주), 1인당 목표 예산(대략), 식이제한/알레르기, 특별히 가고 싶은 장소나 제외하고 싶은 것 등 확인.

Step 2: 제약 조건 수집
- 일정의 유연성(날짜 변경 가능 여부), 도착·출발 시간(공항 도착/출발 시간), 이동수단 제약(짐 많음/대중교통 선호), 체력·이동 제한 등 확인.

Step 3: 예산 레벨 및 우선순위 설정
- 사용자가 제시한 예산을 바탕으로 ‘저가/중간/고급’ 예산 시나리오(1인당 대략치)를 제안하고 우선순위(맛집 우선/관광지 우선/숙소 등)를 확정받기.

Step 4: 지역별 관광지 후보 수집 및 동선화
- 서울 주요 지역(종로·경복궁·북촌, 명동·남대문, 홍대·합정·연남, 이태원·한남, 강남·압구정, 잠실 등)에서 방문 가치 높은 관광지 리스트 작성.
- 하루 동선 효율을 고려해 지역별로 묶을 초안 생성(예: 1일차 종로권, 2일차 홍대+이태원 등).

Step 5: 식당(맛집) 후보 조사
- 각 식사(아침/점심/저녁)별로 지역 내 2~3개 옵션 선정: 음식 종류, 대표메뉴, 가격대, 영업시간, 예약 필요 여부, 주소·전화·리뷰 요약.
- 사용자 취향(한식·해산물·구이·비건 등)에 맞춘 우선순위 목록 준비.

Step 6: 숙소 옵션 선정
- 선호 지역 기준으로 예산에 맞는 숙소 3곳(호텔/게스트하우스/에어비앤비 등) 추천 후보 수집: 1박 가격대, 체크인/체크아웃 시간, 지하철 접근성, 장단점 비교 표 작성.

Step 7: 교통 및 이동 계획 수립
- 공항 ↔ 도심 이동 옵션(AREX/공항버스/택시), 지하철·버스·택시 예상 비용 및 소요시간 계산.
- 하루 동선별 이동수단과 예상 소요시간, 환승 정보 포함.

Step 8: 입장료·체험 비용 및 운영시간 확인
- 각 관광지의 입장료(성인/청소년/어

### TODO 3: Worker Node 구현

- **요구사항**: Planner의 계획을 받아 각 단계를 실행하는 Worker Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["plan"]` (str): Planner가 생성한 계획
    - `state["user_request"]` (str): 원본 사용자 요청
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `result` (str): 계획의 각 단계를 실행한 구체적인 결과
    - `messages` (List[dict]): 메시지 로그에 Worker 결과 추가
- **예상 결과**: 각 Step에 대한 구체적인 정보가 포함된 결과물 (날짜별 일정, 맛집 목록 등)
- **확인 방법**: `print(final_state['result'][:500])`으로 결과 일부 확인
- **힌트**: State에서 plan을 읽어 실행하고 result를 업데이트합니다.

In [8]:
# TODO: Worker Node 구현

def worker_node(state: AgentState) -> AgentState:
    """
    Worker Node: Planner의 계획을 받아 각 단계를 실행합니다.
    
    Args:
        state: 현재 AgentState (plan 포함)
    
    Returns:
        업데이트된 AgentState (result 필드 추가)
    """
    plan = state["plan"]
    user_request = state["user_request"]
    
    # TODO: Worker 프롬프트를 작성하세요
    # - Planner의 계획을 받아 각 단계를 실행하도록 지시
    # - 각 단계별 구체적인 정보를 포함하도록 요청
    worker_prompt = f"""
    # TODO: 프롬프트 작성
    당신은 계획을 실행하는 Worker Agent입니다.

    Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요.
    각 단계별로 구체적인 정보를 포함하세요.
    
    ---
    원본 요청: {user_request}
    
    실행할 계획:
    {plan}
    
    ---
    위 계획을 단계별로 실행하고, 각 단계의 결과를 자세히 작성하세요.
    """
    
    # LLM 호출
    response = llm.invoke(worker_prompt)
    result = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Worker",
        "receiver": "User",
        "content": result,
        "task_type": "result"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "result": result,  # TODO: result 변수로 교체
        "messages": messages
    }

# TODO 완료 후 테스트
final_state = worker_node(result_state)
if final_state['result'] is not None:
    print("=== Worker Node 결과 ===")
    print(f"실행 결과:\n{final_state['result'][:500]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Worker Node 결과 ===
실행 결과:
알겠습니다. 작업을 시작하겠습니다 — 우선 Step 1(사용자 기본 정보)을 알려 주세요. 아래 항목을 복사해 빈칸에 입력해 주시면, 그 정보를 바탕으로 다음 단계(제약 조건 수집)로 진행하겠습니다.

1) 여행 날짜(출발일 — 도착일) 및 시간(가능하면 출발/도착 시간도)  
2) 인원 수(총 인원, 성인/어린이/유아/노인 구분)  
3) 연령대/특이사항(어린이·노약자 동행 여부, 임신 등)  
4) 출발지(예: 인천공항/김포공항/서울역/기타) 및 귀국/복귀 출발지(같은지 여부)  
5) 숙박 선호지역(예: 종로·명동·홍대·강남·잠실 등, 또는 중심지 무관)  
6) 여행 스타일(느긋/빡빡/쇼핑 위주/미식 위주/가족친화 등)  
7) 1인당 목표 예산(대략, KRW 기준이면 편함)  
8) 식이제한·알레르기(비건/채식/해산물 알레르기 등)  
9) 특별히 꼭 가고 싶은 장소(또는 절대 가기 싫은 장소)  
10) 예약 선호(식당/입장권 사전예약 원함 여부)  
11) 최종 산출물 ...


### Guided Build: LangGraph 워크플로우 구성

이제 Planner Node와 Worker Node를 **StateGraph**로 연결하여 워크플로우를 구성합니다.

In [9]:
from langgraph.graph import StateGraph, START, END

# LangGraph 워크플로우 정의 (Guided Build - 코드 제공)
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)

# 엣지 연결: planner → worker → END
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "worker")
workflow.add_edge("worker", END)

# 컴파일
app = workflow.compile()

print("LangGraph 워크플로우 구성 완료!")
print("흐름: planner → worker → END")

LangGraph 워크플로우 구성 완료!
흐름: planner → worker → END


### Test: LangGraph Multi-Agent 실행

컴파일된 워크플로우를 실행하여 Planner → Worker 협업을 테스트합니다.

**확인 기준**:
- [ ] Planner가 3~5개의 명확한 Step을 생성했는가?
- [ ] Worker가 각 Step을 구체적인 정보로 실행했는가?
- [ ] State를 통해 plan이 Worker에게 전달되었는가?
- [ ] 단일 Agent 결과 대비 체계성이 향상되었는가?

**Agent 간 메시지 흐름 확인**:
```python
# messages 리스트에서 흐름 확인
for msg in result["messages"]:
    print(f"[{msg['sender']}] → [{msg['receiver']}]: {msg['task_type']}")
# 예상 출력:
# [Planner] → [Worker]: plan
# [Worker] → [User]: result
```

**체감 포인트**: "LangGraph로 계획과 실행을 분리하니 워크플로우가 명확하고 결과 품질도 높아졌네!"

In [10]:
# LangGraph 워크플로우 실행
def run_multi_agent(user_request: str) -> dict:
    """
    LangGraph Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 LangGraph Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    
    final_state = app.invoke(initial_state)
    
    print("✅ 실행 완료\n")
    
    return final_state

# 테스트 실행
workflow_result = run_multi_agent("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 최종 결과")
print("=" * 50)
print(workflow_result["result"])

🚀 LangGraph Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
✅ 실행 완료

📊 최종 결과
알겠습니다. 저는 Worker Agent로서 제시된 15단계 계획을 순서대로 실행하겠습니다. 다만 일부 단계는 사용자의 구체 정보(출발일/복귀일, 출발지, 동행인 등)가 있어야 정확한 조사·견적이 가능하므로, 먼저 Step 1·2에서 요청하는 정보를 받아야 맞춤 결과를 완성할 수 있습니다. 아래는 각 단계별 실행 결과(초안)와 사용자에게 요청하는 항목을 정리한 내용입니다. 가능한 한 상세히 준비했으니, Step 1·2 답변을 주시면 바로 맞춤 조사·최종안 작성으로 넘어가겠습니다.

Step 1 — 사용자 정보 수집 (실행 결과: 질문 목록 작성 및 사용자 요청)
- 요청 항목(답변을 주세요):
  1. 여행 날짜: 출발일(YYYY-MM-DD) / 복귀일(YYYY-MM-DD) — 혹은 유연한 범위
  2. 출발지(예: 거주지 주소/도시/역·공항): 어디서 출발하시는지
  3. 동행인 구성: 인원수 및 연령대(성인 몇 명 / 어린이 몇 명(나이) / 노약자 여부)
  4. 여행 페이스(선호 속도): 여유(느긋) / 보통 / 빠름(빡빡)
  5. 이동수단 선호(우선순위): 자가용 / KTX/무궁화/버스 / 항공 / 상관없음
  6. 숙소 유형 선호: 게스트하우스 / 비즈니스 호텔 / 호텔(중급/고급) / Airbnb / 상관없음
  7. 숙소 위치 선호(예: 명동/홍대/강남/이태원/종로/서울역 등)
  8. 식사 성향: 가성비·현지음식 / 맛집 탐방 / 고급식당 / 특수식(채식/알레르기)
  9. 주요 관심사: 문화(박물관/전시)/역사/쇼핑/맛집/야경/카페/체험활동 등 (우선순위)
  10. 여행 목적(예: 휴식/데이트/가족여행/친구 모임/개인 관람 등)
  11. 예산 우선순위(모든 항목 동등 / 숙소优先 / 교통优先 / 식비优先 등)
  12. 특별 요구사항: 이동 편의(휠

---

# Step 5: Reflection 패턴 추가

### Concept Check: Reflection 패턴이란?

**왜 이 단계가 필요한가요?**  
Planner-Worker만으로도 체계적인 결과를 얻을 수 있지만, 결과의 품질을 한 단계 더 높이려면 자기 검토가 필요합니다. Reflection 패턴은 "결과가 요청을 충족하는가?"를 스스로 평가하고 개선하는 **루프**입니다.

**Reflection 패턴**은 Agent가 자신의 결과를 검토하고 개선하는 자기 검토 루프입니다.

#### Reflection 루프 구조 (핵심)

```
[Planner] → [Worker] → [Reflection] ─── 품질 충분? ──→ [END]
                ↑                              │
                └──── 개선 필요 (재실행) ────────┘
```

**핵심 포인트**: Reflection은 단순 평가가 아니라 **평가→개선 루프**입니다.
- Reflection Agent가 결과를 검토하고 "충족/부분 충족/미충족"을 판정
- "부분 충족" 또는 "미충족"이면 개선된 결과를 만들어 Worker에게 다시 보냄
- 무한 루프 방지를 위해 `reflection_count`로 최대 반복 횟수를 제한 (예: 최대 1회)

#### Reflection Agent의 역할
- 결과가 원본 요청을 충족하는지 검토
- 누락된 정보나 개선 사항 식별
- **충족 시**: 최종 결과 확정 → END
- **미충족 시**: 개선 피드백을 result에 반영 → Worker 재실행

#### Agent 간 메시지 흐름 (Reflection 포함)
```
[Worker] State 쓰기: result = "Day 1: 경복궁..."
    ↓
[Reflection] State 읽기: user_request, plan, result
    ↓ LLM 호출 (검토)
[Reflection] 판정:
    충족 → State 쓰기: final_result = result → END
    부분 충족 → State 쓰기: result = "개선된 결과..." → Worker 재실행
```

### TODO 4: Reflection Agent 추가

- **요구사항**: Worker의 결과를 검토하고 개선 제안을 하는 Reflection Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["result"]` (str): Worker가 생성한 결과
    - `state["user_request"]` (str): 원본 사용자 요청
    - `state["plan"]` (str): Planner가 생성한 계획
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `reflection` (str): 검토 결과 (충족 여부, 강점, 개선 필요)
    - `final_result` (str): 개선된 최종 결과
    - `messages` (List[dict]): 메시지 로그에 Reflection 결과 추가
- **예상 결과**: 
  - 검토 결과: "충족/부분 충족/미충족" 판정 + 구체적 피드백
  - 최종 결과: Worker 결과를 보완한 완성도 높은 결과물
- **확인 방법**: `print(reflection_state['reflection'][:800])`으로 검토 결과 확인
- **힌트**: 결과를 검토하고 누락된 부분이나 개선점을 제안합니다. 프롬프트에 "검토 항목"과 "출력 형식"을 명시하세요.

In [11]:
# TODO: Reflection Node 구현

MAX_REFLECTIONS = 1  # 최대 Reflection 반복 횟수 (무한 루프 방지)

def reflection_node(state: AgentState) -> AgentState:
    """
    Reflection Node: Worker의 결과를 검토하고 개선 제안을 합니다.
    
    Args:
        state: 현재 AgentState (result 포함)
    
    Returns:
        업데이트된 AgentState (reflection, final_result 필드 추가)
    """
    result = state["result"]
    user_request = state["user_request"]
    plan = state["plan"]
    count = state.get("reflection_count", 0)
    
    # TODO: Reflection 프롬프트를 작성하세요
    # - Worker의 결과를 검토하고 원본 요청 충족 여부 평가
    # - 검토 항목: 요구사항 충족, 누락 정보, 정확성, 개선 필요 사항
    # - 출력 형식: ## 검토 결과 + ## 최종 개선 결과
    reflection_prompt = f"""
    # TODO: 프롬프트 작성
    당신은 결과를 검토하는 Reflection Agent입니다.
    
    Worker Agent가 생성한 결과를 검토하고, 원본 요청을 충족하는지 평가하세요.
    
    검토 항목:
    1. 원본 요청의 모든 요구사항이 충족되었는가?
    2. 누락된 정보가 있는가?
    3. 정보의 정확성과 구체성은 적절한가?
    4. 개선이 필요한 부분이 있는가?
    
    ---
    원본 요청: {user_request}
    
    실행된 계획:
    {plan}
    
    Worker의 결과:
    {result}
    ---
    위 내용을 검토하고 다음 형식으로 작성하세요:
    
    ## 검토 결과
    - 충족 여부: [충족/부분 충족/미충족]
    - 강점: [잘된 부분]
    - 개선 필요: [부족한 부분]
    
    ## 최종 개선 결과
    [Worker의 결과를 보완하여 최종 결과 작성]
    """
    
    # LLM 호출
    response = llm.invoke(reflection_prompt)
    reflection = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Reflection",
        "receiver": "User",
        "content": reflection,
        "task_type": "reflection"
    })
    
    # TODO: State 업데이트하여 반환 (reflection_count 증가 포함)
    return {
        **state,
        "reflection": reflection,  # TODO: reflection 변수로 교체
        "final_result": reflection,  # TODO: reflection 변수로 교체
        "reflection_count": state.get("reflection_count", 0) + 1,
        "messages": messages
    }

def should_continue_reflection(state: AgentState) -> str:
    """Reflection 후 종료할지 Worker로 돌아갈지 판단"""
    count = state.get("reflection_count", 0)
    reflection = state.get("reflection", "")
    
    # 최대 반복 횟수 초과 시 종료
    if count >= MAX_REFLECTIONS:
        return "end"
    
    # "충족" 판정이면 종료, 아니면 Worker로 재실행
    if "충족 여부: 충족" in reflection:
        return "end"
    
    return "worker"

# TODO 완료 후 테스트
reflection_state = reflection_node(final_state)
if reflection_state['reflection'] is not None:
    print("=== Reflection Node 결과 ===")
    print(f"검토 결과:\n{reflection_state['reflection'][:800]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Reflection Node 결과 ===
검토 결과:
## 검토 결과
- 충족 여부: 부분 충족
- 강점:
  - 전체 일정 수립을 위한 매우 체계적이고 상세한 단계(1~15)를 제시하여 계획의 흐름이 명확함.
  - 맛집·숙소·교통·예산·예약·지도 등 중요한 항목을 모두 고려한 체크리스트 형태로 준비되어 있음.
  - 사용자 정보 수집을 위한 항목(질문지)이 구체적이라 맞춤형 일정 작성에 유용함.
- 개선 필요:
  - 원본 요청(“3일 서울 여행 계획 — 맛집, 관광지, 예산 포함”)에 대한 즉시 3일치 예시 일정과 예산 제시가 없음. 사용자의 응답을 기다리는 상태만 제시되어 있어 즉시 제공을 기대한 사용자 요구를 충족하지 못함.
  - 실제 맛집·관광지·예산(금액 수치) 예시가 포함되어 있지 않음.
  - 사용자가 지금 당장 정보를 주지 않더라도 기본 가정(예: 1인/중간 예산/인천 도착 등)을 두고 샘플 일정을 제시하면 더 친절함.

## 최종 개선 결과
(아래는 사용자 추가 정보가 없다는 가정 하에, 일반적인 1인 기준·중간(보통) 예산·인천공항 도착·명동/종로권 숙박 선호를 전제로 작성한 3일 샘플 일정과 예산안입니다. 원하시면 이 가정들을 바꿔 재작성해 드립니다.)

가정
- 인원: 1인 (성인)
- 숙박: 중급 호텔(명동/종로권), 2박
- 도착/출발: 인천공항 (도착: 오전 첫날, 출발: 셋째날 저녁)
- 여행 스타일: 적당한 관광·맛집 위주(무리하지 않는 일정)
- 이동: 대중교통(지하철/버스) 우선

3일 일정 요약
Day 1 (종로·경복궁·북촌·명동)
- 오전
  - 경복궁 관람 (입장 09:00~) — 관람 1.5시간
  - 국립민속박물관...


### TODO 5: 성과 평가 함수 구현

- **요구사항**: Multi-Agent 시스템의 결과를 평가하는 함수를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리 (user_request, plan, result, reflection 포함)
- **출력**: 
  - `dict`: 평가 결과
    - `score` (int): 종합 점수 (0~100)
    - `feedback` (str): 전체 피드백
    - `details` (dict): 세부 평가 항목별 점수
- **평가 항목**:
  - Planner 평가: 계획의 완성도, 단계 적절성 (0~30점)
  - Worker 평가: 실행 품질, 정보 충실도 (0~40점)
  - 전체 평가: 사용자 요청 충족도 (0~30점)
- **예상 결과**: 
  - 종합 점수: 70~90점 범위 (Reflection 적용 후 품질 향상)
  - JSON 출력 예시:
    ```json
    {
      "planner_score": 25,
      "worker_score": 35,
      "overall_score": 25,
      "feedback": "계획이 체계적이며, 대부분의 요청 사항을 충족함"
    }
    ```
- **확인 방법**: `print(f"종합 점수: {evaluation['score']}/100")`로 점수 출력
- **힌트**: LLM에게 평가 기준과 State 정보를 제공하여 점수와 피드백을 생성합니다. JSON 파싱 시 `re.search()`로 JSON 부분만 추출합니다.

In [12]:
# TODO: 성과 평가 함수 구현
# 아래 함수를 완성하세요.

import json
import re

def evaluate_multi_agent(state: AgentState) -> dict:
    """
    Multi-Agent 시스템의 성과를 평가합니다.
    
    Args:
        state: 현재 AgentState (user_request, plan, result, reflection 포함)
    
    Returns:
        dict: {"score": int, "feedback": str, "details": dict}
    """

    # 평가 프롬프트 구성
    eval_prompt = f"""
당신은 Multi-Agent 시스템의 성과를 평가하는 평가자입니다.

## 평가 기준
1. Planner 평가 (0~30점): 계획의 완성도, 단계 구성의 적절성
2. Worker 평가 (0~40점): 실행 품질, 정보의 충실도와 정확성
3. 전체 평가 (0~30점): 사용자 요청 충족도, 협업 효율성

## 입력 정보
- 사용자 요청: {state['user_request']}
- Planner의 계획: {state['plan']}
- Worker의 결과: {state['result']}
- Reflection 검토: {state.get('reflection', '없음')}

## 출력 형식
반드시 아래 JSON 형식만 출력하세요. 다른 텍스트는 포함하지 마세요.
{{"planner_score": <0~30>, "worker_score": <0~40>, "overall_score": <0~30>, "feedback": "<전체 피드백>"}}
"""

    response = llm.invoke(eval_prompt)
    content = response.content

    # JSON 파싱 (마크다운 코드 블록 처리)
    try:
        # 마크다운 코드 블록에서 JSON 추출 (```json ... ``` 또는 ``` ... ```)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # 코드 블록이 없으면 중괄호로 시작하는 JSON 추출
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content

        result = json.loads(json_str)

        # TODO: result를 이용하여 점수를 합산하고 결과를 반영하세요
        total_score = result['planner_score'] + result['worker_score'] + result['overall_score']          # TODO: 세 항목의 점수를 합산하세요

        return {
            "score": total_score,
            "feedback": result['feedback'],       # TODO: result의 feedback 값으로 교체하세요
            "details": {
                "planner": result['planner_score'],    # TODO: result의 planner_score 값으로 교체하세요
                "worker": result['worker_score'],     # TODO: result의 worker_score 값으로 교체하세요
                "overall": result['overall_score']     # TODO: result의 overall_score 값으로 교체하세요
            }
        }
    except Exception as e:
        return {"score": 0, "feedback": f"평가 실패: {str(e)}", "details": {}}

# 테스트
evaluation = evaluate_multi_agent(reflection_state)
if evaluation['feedback'] is not None:
    print("=== 성과 평가 결과 ===")
    print(f"종합 점수: {evaluation['score']}/100")
    print(f"세부 점수: Planner {evaluation['details'].get('planner', 0)}/30, Worker {evaluation['details'].get('worker', 0)}/40, 전체 {evaluation['details'].get('overall', 0)}/30")
    print(f"피드백: {evaluation['feedback']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== 성과 평가 결과 ===
종합 점수: 79/100
세부 점수: Planner 28/30, Worker 27/40, 전체 24/30
피드백: 강점: Planner는 15단계로 매우 체계적이고 세부 항목(맛집·숙소·교통·예산·예약 등)을 포괄적으로 정리해 일정 수립 흐름이 명확합니다. Worker는 사용자 응답을 유도하는 구체적 질문지를 제공해 맞춤형 일정 수집에 적합합니다. 개선점: 원본 사용자 요청(즉시 3일 샘플 일정과 예산 제시)에 대해 초반에 기본 가정으로 만든 샘플 일정을 직접 제시하는 적극성이 부족했습니다. Planner는 전반적으로 완성도가 높았으나, 사용자 정보를 기다리는 상황에서도 임의 가정(예: 1인·중간예산·인천도착 등)으로 빠른 샘플을 자동 제공하는 옵션을 포함하면 더 친절합니다. Worker는 질문지 제공 외에 기본 가정으로 만든 예시 일정·예산을 즉시 제시하거나, '정보 일부만 있어도 가능' 등 선택지를 제안하면 응답률과 사용자 만족도가 올라갑니다. 권장사항: 향후 동일 작업 시 초기 응답에 '기본 가정 하 샘플 일정'을 항상 포함하고, 예약·주소·영업시간 등 구체 데이터(가능 시 링크)를 병행 제공하세요. 또한 단계별 산출물을 어떤 형식(PDF/구글문서)으로 받을지 초기에 확인하면 최종 전달이 원활합니다.


### Guided Build: Reflection 패턴 워크플로우 구성

이제 Planner → Worker → Reflection 순서로 워크플로우를 구성합니다.
Reflection은 `should_continue_reflection` 함수로 **조건 분기**합니다:
- 품질 충분 또는 최대 반복 도달 → END
- 개선 필요 → Worker로 재실행

In [13]:
# Reflection 패턴 워크플로우 구성
workflow_reflection = StateGraph(AgentState)

# 노드 추가
workflow_reflection.add_node("planner", planner_node)
workflow_reflection.add_node("worker", worker_node)
workflow_reflection.add_node("reflection", reflection_node)

# 엣지 연결: planner → worker → reflection
workflow_reflection.add_edge(START, "planner")
workflow_reflection.add_edge("planner", "worker")
workflow_reflection.add_edge("worker", "reflection")

# Reflection 후 조건 분기: END 또는 Worker 재실행
workflow_reflection.add_conditional_edges(
    "reflection",
    should_continue_reflection,
    {"end": END, "worker": "worker"}
)

# 컴파일
app_reflection = workflow_reflection.compile()

print("Reflection 패턴 워크플로우 구성 완료!")
print("흐름: planner → worker → reflection → (조건 분기) → END 또는 worker")

Reflection 패턴 워크플로우 구성 완료!
흐름: planner → worker → reflection → (조건 분기) → END 또는 worker


In [14]:
# Reflection 패턴 워크플로우 실행
def run_multi_agent_reflection(user_request: str) -> dict:
    """
    Reflection 패턴이 포함된 Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 Reflection Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    print("🔍 [Reflection Agent] 결과 검토 중...")
    
    final_state = app_reflection.invoke(initial_state)
    
    print(f"\n✅ 실행 완료 (Reflection 횟수: {final_state['reflection_count']})\n")
    
    return final_state

# 테스트 실행
reflection_result = run_multi_agent_reflection("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 Reflection 최종 결과")
print("=" * 50)
print(reflection_result["final_result"])

🚀 Reflection Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
🔍 [Reflection Agent] 결과 검토 중...

✅ 실행 완료 (Reflection 횟수: 1)

📊 Reflection 최종 결과
## 검토 결과
- 충족 여부: 부분 충족
- 강점:
  - 매우 체계적이고 단계별로 필요한 작업을 상세히 정리함(16단계 전체 구성).
  - 예산 초안(카테고리별 배분), 일별 샘플 동선, 숙소·교통·식사·비상대책 등 실무에 필요한 체크리스트를 폭넓게 포함함.
  - 사용자 입력(출발지·날짜·우선순위 등)을 명확하게 요청하여 개인화가 가능하도록 설계함.
- 개선 필요:
  - 원본 요청(서울 2박3일 여행 계획 + 예산 50만원 이내)에 대해 즉시 사용할 수 있는 구체적·완성된 플랜(예: 1인 기준의 일정, 숙소 후보, 구체적 비용 합계)은 제공되지 않음 — 대부분은 '사용자 입력을 받은 뒤 실행'하는 절차·지침 수준임.
  - 예시로 제시된 비용·일정은 있지만 출발지 미확인 상태에서의 구체적 비용(교통비 등)·숙소 후보(구체명, 가격 범위)·예약 권장 시기 등 실무적 수치가 부족함.
  - 예산 합계는 제시되었지만, 실제 명소 입장료·숙박 실물 견적·예상 식비 합산 등 검증 가능한 금액 합산 샘플 플랜이 없음.
  - 사용자에게 바로 제공 가능한 '완성된 2박3일 일정(인쇄용/예산표 포함)' 파일 형태 결과물(간단한 요약본)은 미제공.

## 최종 개선 결과
(가정: 1인 기준 / 출발지: 서울시내(또는 서울 도착이 이미 해결된 상태) / 총예산 500,000원 한도)
원하시면 출발지(다른 도시 출발 시 KTX/버스/항공 포함)와 우선순위(관광/맛집/쇼핑/여유)를 알려주시면 바로 맞춤 수정합니다. 아래는 바로 쓸 수 있는 샘플 플랜(예산 내)입니다.

1) 요약(가정)
- 인원: 1인
- 일정: 서울 2박3일 (예: 금 오후 도착 ~ 일 점심 출발)
- 

### Test: Reflection 패턴 실행 결과 확인

**확인 기준**:
- [ ] Reflection Agent가 Worker의 결과를 검토했는가?
- [ ] 충족 여부 판정이 이루어졌는가?
- [ ] 개선이 필요한 경우 Worker가 재실행되었는가?
- [ ] 최종 결과에 개선 내용이 반영되었는가?

**체감 포인트**: "Reflection Agent가 자동으로 결과를 검토하고 품질을 높여주니, 수동 검토 없이도 신뢰할 수 있는 결과를 얻을 수 있구나!"

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### API 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `AuthenticationError` | API Key 미설정 또는 잘못된 키 | `.env` 파일에 `GMS_KEY` 확인 |
| `RateLimitError` | API 호출 한도 초과 | 잠시 대기 후 재실행 (1~2분) |
| `InvalidRequestError` | 토큰 제한 초과 | 입력 텍스트 길이 줄이기 또는 청킹 크기 조정 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |
| ChromaDB `sqlite3` 오류 | SQLite 버전 낮음 | `%pip install pysqlite3-binary` 후 커널 재시작 |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 출력이 비어있음 | 환경 변수 로드 실패 | `.env` 파일 경로 확인 후 커널 재시작 |


---

## 실습 마무리

### 학습 내용 정리

이번 실습에서 배운 핵심 내용:

1. **단일 Agent의 한계**: 복잡한 작업에서 혼란, 누락, 품질 저하 발생
2. **Multi-Agent 대표 패턴**: Planner-Worker, Supervisor, Debate, Reflection, Hierarchical
3. **Planner-Worker 패턴**: 계획 수립과 실행을 분리하여 체계적 결과 도출
4. **Reflection 패턴**: 자기 검토 루프로 결과 품질 향상
5. **LangGraph StateGraph**: 상태 기반 워크플로우로 에이전트 간 협업 구현

### 학생용 자가 체크리스트

- [ ] 단일 Agent의 한계를 직접 체험했는가?
- [ ] Multi-Agent 대표 패턴 5가지를 설명할 수 있는가?
- [ ] LangGraph State를 정의하고 이해했는가?
- [ ] Planner Node가 계획을 수립하는 코드를 작성했는가?
- [ ] Worker Node가 계획을 실행하는 코드를 작성했는가?
- [ ] Reflection Node가 결과를 검토하고 개선하는 코드를 작성했는가?
- [ ] Reflection 패턴 전후 결과 품질 차이를 확인했는가?
- [ ] 성과 평가 함수로 결과 품질을 정량적으로 측정했는가?

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.